In [ ]:
import os
import sys
import time
import json
import random

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from src.data_utils import get_mnist_dataloaders, get_class_names
from src.models import AblationCNN, count_parameters
from src.train import fit_model
from src.evaluate import collect_predictions, compute_classification_metrics
from src.visualize import plot_training_history, plot_confusion_matrix

In [ ]:
SEED = 42
BATCH_SIZE = 64
EPOCHS = 8
LR = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULT_TABLE_DIR = "../results/tables"
RESULT_FIG_DIR = "../results/figures/g2_ablation"
os.makedirs(RESULT_TABLE_DIR, exist_ok=True)
os.makedirs(RESULT_FIG_DIR, exist_ok=True)

print("DEVICE:", DEVICE)

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
class_names = get_class_names()

In [ ]:
def run_single_ablation_experiment(
    exp_name: str,
    channels=(8, 16),
    use_pool=True,
    augment=False,
    normalize=False,
    batch_size=64,
    epochs=8,
    lr=1e-3,
    seed=42,
):
    set_seed(seed)

    train_loader, val_loader, test_loader = get_mnist_dataloaders(
        data_dir="../data",
        batch_size=batch_size,
        val_size=5000,
        num_workers=0,
        augment=augment,
        normalize=normalize,
        seed=seed,
    )

    model = AblationCNN(
        channels=channels,
        use_pool=use_pool,
        hidden_dim=64,
        num_classes=10,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    start_time = time.time()
    result = fit_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        epochs=epochs,
    )
    train_time = time.time() - start_time

    logits, preds, labels = collect_predictions(result["model"], test_loader, DEVICE)
    metrics = compute_classification_metrics(labels, preds, class_names=class_names)

    record = {
        "exp_name": exp_name,
        "channels": str(channels),
        "use_pool": use_pool,
        "augment": augment,
        "normalize": normalize,
        "epochs": epochs,
        "lr": lr,
        "param_count": count_parameters(model),
        "best_val_acc": result["best_val_acc"],
        "test_accuracy": metrics["accuracy"],
        "test_macro_precision": metrics["macro_precision"],
        "test_macro_recall": metrics["macro_recall"],
        "test_macro_f1": metrics["macro_f1"],
        "train_time_sec": train_time,
        "history": result["history"],
        "confusion_matrix": metrics["confusion_matrix"],
        "classification_report": metrics["classification_report"],
        "model": result["model"],
    }

    return record

In [ ]:
ablation_plan = [
    {
        "exp_name": "baseline_pool_ch_8_16",
        "channels": (8, 16),
        "use_pool": True,
        "augment": False,
        "normalize": False,
    },
    {
        "exp_name": "no_pool_ch_8_16",
        "channels": (8, 16),
        "use_pool": False,
        "augment": False,
        "normalize": False,
    },
    {
        "exp_name": "pool_ch_16_32",
        "channels": (16, 32),
        "use_pool": True,
        "augment": False,
        "normalize": False,
    },
    {
        "exp_name": "pool_ch_32_64",
        "channels": (32, 64),
        "use_pool": True,
        "augment": False,
        "normalize": False,
    },
    {
        "exp_name": "pool_ch_8_16_aug",
        "channels": (8, 16),
        "use_pool": True,
        "augment": True,
        "normalize": False,
    },
]

In [ ]:
all_results = []

for cfg in ablation_plan:
    print("=" * 100)
    print("Running:", cfg["exp_name"])
    record = run_single_ablation_experiment(
        exp_name=cfg["exp_name"],
        channels=cfg["channels"],
        use_pool=cfg["use_pool"],
        augment=cfg["augment"],
        normalize=cfg["normalize"],
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        lr=LR,
        seed=SEED,
    )
    all_results.append(record)

    print(f"Done: {cfg['exp_name']}")
    print(f"Params      : {record['param_count']}")
    print(f"Best Val Acc: {record['best_val_acc']:.4f}")
    print(f"Test Acc    : {record['test_accuracy']:.4f}")
    print(f"Macro F1    : {record['test_macro_f1']:.4f}")
    print(f"Time (sec)  : {record['train_time_sec']:.2f}")

In [ ]:
summary_df = pd.DataFrame([
    {
        "exp_name": r["exp_name"],
        "channels": r["channels"],
        "use_pool": r["use_pool"],
        "augment": r["augment"],
        "normalize": r["normalize"],
        "param_count": r["param_count"],
        "best_val_acc": r["best_val_acc"],
        "test_accuracy": r["test_accuracy"],
        "test_macro_precision": r["test_macro_precision"],
        "test_macro_recall": r["test_macro_recall"],
        "test_macro_f1": r["test_macro_f1"],
        "train_time_sec": r["train_time_sec"],
    }
    for r in all_results
])

summary_df = summary_df.sort_values(by="test_accuracy", ascending=False).reset_index(drop=True)
summary_df

In [ ]:
summary_csv_path = os.path.join(RESULT_TABLE_DIR, "g2_ablation_summary.csv")
summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

summary_md_path = os.path.join(RESULT_TABLE_DIR, "g2_ablation_summary.md")
with open(summary_md_path, "w", encoding="utf-8") as f:
    f.write(summary_df.to_markdown(index=False))

print("Saved:", summary_csv_path)
print("Saved:", summary_md_path)

In [ ]:
plt.figure(figsize=(10, 6))

for r in all_results:
    val_acc = r["history"]["val_acc"]
    epochs_axis = np.arange(1, len(val_acc) + 1)
    plt.plot(epochs_axis, val_acc, marker="o", label=r["exp_name"])

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("G2 Ablation: Validation Accuracy Curves")
plt.legend()
plt.tight_layout()

save_path = os.path.join(RESULT_FIG_DIR, "g2_val_acc_comparison.png")
plt.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

In [ ]:
plot_df = summary_df.copy()

plt.figure(figsize=(10, 5))
plt.bar(plot_df["exp_name"], plot_df["test_accuracy"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Test Accuracy")
plt.title("G2 Ablation: Test Accuracy Comparison")
plt.tight_layout()

save_path = os.path.join(RESULT_FIG_DIR, "g2_test_accuracy_bar.png")
plt.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

In [ ]:
plot_df = summary_df.copy()

plt.figure(figsize=(10, 5))
plt.bar(plot_df["exp_name"], plot_df["test_macro_f1"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Macro F1")
plt.title("G2 Ablation: Test Macro F1 Comparison")
plt.tight_layout()

save_path = os.path.join(RESULT_FIG_DIR, "g2_test_macro_f1_bar.png")
plt.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(summary_df["param_count"], summary_df["test_accuracy"], s=80)

for _, row in summary_df.iterrows():
    plt.annotate(
        row["exp_name"],
        (row["param_count"], row["test_accuracy"]),
        textcoords="offset points",
        xytext=(5, 5),
    )

plt.xlabel("Parameter Count")
plt.ylabel("Test Accuracy")
plt.title("G2 Ablation: Parameter Count vs Test Accuracy")
plt.tight_layout()

save_path = os.path.join(RESULT_FIG_DIR, "g2_param_vs_acc.png")
plt.savefig(save_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", save_path)

In [ ]:
best_idx = summary_df["test_accuracy"].idxmax()
best_exp_name = summary_df.loc[best_idx, "exp_name"]
best_result = next(r for r in all_results if r["exp_name"] == best_exp_name)

print("Best experiment:", best_exp_name)
print(best_result["classification_report"])

plot_confusion_matrix(
    best_result["confusion_matrix"],
    class_names=class_names,
    save_path=os.path.join(RESULT_FIG_DIR, f"{best_exp_name}_confusion_matrix.png"),
)

In [ ]:
def make_conclusion_text(summary_df: pd.DataFrame) -> str:
    best_row = summary_df.sort_values("test_accuracy", ascending=False).iloc[0]
    worst_row = summary_df.sort_values("test_accuracy", ascending=True).iloc[0]

    text = f"""
G2 消融实验结论草稿：

1. 最优模型为 {best_row['exp_name']}，测试准确率为 {best_row['test_accuracy']:.4f}，Macro-F1 为 {best_row['test_macro_f1']:.4f}。
2. 最弱模型为 {worst_row['exp_name']}，测试准确率为 {worst_row['test_accuracy']:.4f}。
3. 比较是否使用池化的实验，可用于分析池化层在降低特征图维度、增强局部平移稳定性方面的作用。
4. 比较不同通道数的实验，可用于分析模型表征能力与参数规模之间的关系，并观察收益递减现象。
5. 数据增强实验可用于分析轻量几何扰动是否提升模型泛化性能；若提升有限，也符合 MNIST 任务本身相对简单的特点。
"""
    return text.strip()

conclusion_text = make_conclusion_text(summary_df)
print(conclusion_text)

with open(os.path.join(RESULT_TABLE_DIR, "g2_conclusion_draft.txt"), "w", encoding="utf-8") as f:
    f.write(conclusion_text)